# Data validation with Voluptuous (schema definitions)

In this notebook we use [Voluptuous](https://github.com/alecthomas/voluptuous) to define schemas for our data. We can then use schema checking at various points in our cleanup to ensure that we meet the criteria. Finally, we can use schema checking exceptions to flag, set aside or remove impure or invalid data.

<div class="alert alert-block alert-info">

**See also:**

* [Validr](https://github.com/guyskk/validr)
* [marshmallow](https://marshmallow.readthedocs.io/en/latest/)
</div>

## 1. Imports

In [1]:
import datetime
import logging

import pandas as pd

from voluptuous import ALLOW_EXTRA, All, Range, Required, Schema
from voluptuous.error import Invalid, MultipleInvalid

* `Required` marks the node of a schema as required and optionally specifies a default value, see also [voluptuous.schema_builder.Required](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.schema_builder.Required).
* `Range` limits the value to a range where either `min` or `max` can be omitted; see also [voluptuous.validators.Range](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.validators.Range).
* `ALL` is used for cross-field validations: checks the basic structure of the data in a first pass and only in the second pass the cross-field validation is applied; see also [voluptuous.validators.All](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.validators.All).
* `ALLOW_EXTRA` allows additional dictionary keys.
* `MultipleInvalid` is based on `Invalid`, see also [voluptuous.error.MultipleInvalid](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.error.MultipleInvalid).
* `Invalid` marks data as invalid, see also [voluptuous.error.Invalid](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.error.Invalid).

## 2. Logger

In [2]:
logger = logging.getLogger(__name__)

## 3. Read sample data

In [3]:
sales = pd.read_csv(
    "https://raw.githubusercontent.com/kjam/data-cleaning-101/master/data/sales_data.csv",
)

## 4. Examine data

In [4]:
sales.head()

,Unnamed: 0,timestamp,city,store_id,sale_number,sale_amount,associate
0,0,2018-09-10 05:00:45,Williamburgh,6,1530,1167.0,Gary Lee
1,1,2018-09-12 10:01:27,Ibarraberg,1,2744,258.0,Daniel Davis
2,2,2018-09-13 12:01:48,Sarachester,2,1908,266.0,Michael Roth
3,3,2018-09-14 20:02:19,Caldwellbury,14,771,-108.0,Michaela Stewart
4,4,2018-09-16 01:03:21,Erikaland,11,1571,-372.0,Mark Taylor


In [5]:
sales.shape

(213, 7)

In [6]:
sales.dtypes

Unnamed: 0       int64
timestamp       object
city            object
store_id         int64
sale_number      int64
sale_amount    float64
associate       object
dtype: object

## 5. Define schema

In the column `sale_amount` all values should be between 2.5 and 1450.99:

In [7]:
schema = Schema(
    {
        Required("sale_amount"): All(float, Range(min=2.50, max=1450.99)),
    },
    extra=ALLOW_EXTRA,
)

To be able to use the elements of one column as keys and the elements of another column as values, we simply make the desired column the index of the DataFrame and transpose it with the function `.T()`; see also [pandas.DataFrame.transpose](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transpose.html).

In [8]:
error_count = 0
for s_id, sale in sales.T.to_dict().items():
    try:
        schema(sale)
    except MultipleInvalid as e:
        logger.warning(
            f"issue with sale: {s_id} ({sale['timestamp']}) - {e}",
        )
        error_count += 1

issue with sale: 3 (2018-09-14 20:02:19) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 4 (2018-09-16 01:03:21) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 5 (2018-09-18 03:04:11) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 6 (2018-09-20 12:04:49) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 7 (2018-09-21 15:05:42) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 10 (2018-09-27 04:07:32) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 13 (2018-10-02 17:08:42) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 15 (2018-10-07 06:09:00) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 19 (2018-10-14 08:10:23) - value must be at least 2.5 for dictionary value @

In [9]:
error_count

69

Currently, however, we do not yet know whether

* we have a wrongly defined schema
* possibly negative values are returned or incorrectly marked
* higher values are combined purchases or special sales

## 6. Adding a custom validation

In [10]:
def valid_date(fmt="%Y-%m-%d %H:%M:%S"):
    return lambda v: datetime.datetime.strptime(v, fmt).replace(
        tzinfo=datetime.timezone.utc
    )

In [11]:
schema = Schema(
    {
        Required("timestamp"): All(valid_date()),
    },
    extra=ALLOW_EXTRA,
)

In [12]:
error_count = 0
for s_id, sale in sales.T.to_dict().items():
    try:
        schema(sale)
    except MultipleInvalid as e:
        logger.warning(
            f"issue with sale: {s_id} ({sale['timestamp']}) - {e}",
        )
        error_count += 1

In [13]:
error_count

0

## 7. Valid date structures are not yet valid dates

In [14]:
def valid_date(fmt="%Y-%m-%d %H:%M:%S"):
    def validation_func(v):
        try:
            assert datetime.datetime.strptime(v, fmt).replace(
                tzinfo=datetime.timezone.utc
            ) <= datetime.datetime.now(tz=datetime.timezone.utc)
        except AssertionError:
            msg = f"The date is in the future! {v}"
            raise Invalid(msg) from AssertionError

    return validation_func

In [15]:
schema = Schema(
    {
        Required("timestamp"): All(valid_date()),
    },
    extra=ALLOW_EXTRA,
)

In [16]:
error_count = 0
for s_id, sale in sales.T.to_dict().items():
    try:
        schema(sale)
    except MultipleInvalid as e:
        logger.warning(
            f"issue with sale: {s_id} ({sale['timestamp']}) - {e}",
        )
        error_count += 1

In [17]:
error_count

0